# AWS Data Platform
**Day 1 — Technology & Architecture Module**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>Core Insight:</strong> AWS has three services that form the backbone of most modern data lakes:
S3 (store), Glue (catalog + transform), Athena (query). You can build a production
data platform with no servers to manage. Understanding how they fit together — and
their cost/performance tradeoffs — is what separates data engineers from data analysts.
</div>

### The Core Stack
```
Raw Data → S3 (data lake)  →  Glue Crawler  →  Glue Data Catalog
                                                       ↓
                                               Athena (SQL query)
                                                       ↓
                                       QuickSight / Jupyter / API Consumer
```

## S3 — The Foundation

S3 is an object store, not a file system. A "file path" like `s3://my-bucket/data/2026/02/27/file.parquet` is a **key string** — there are no actual folders, just key prefixes.

### Partitioning — Your First Performance Decision

```
s3://telemetry-lake/
  metrics/
    year=2026/
      month=02/
        day=27/
          part-0001.parquet
          part-0002.parquet
        day=28/
          ...
```

When Athena queries `WHERE year='2026' AND month='02'`, it skips all other partitions entirely. A 365-day dataset queried for 7 days = Athena reads ~2% of the data.

**Partition key rule:** Partition by the columns you filter on most. Date is almost always first. Environment (prod/dev), region, or customer tier often follow.

### Storage Classes (Cost vs Access Speed)

| Class | Use When | Retrieval |
|-------|---------|-----------|
| Standard | Hot data, frequent access | Instant |
| Standard-IA | 30+ days old, infrequent access | Instant, higher per-read |
| Glacier Instant | 90+ days old, occasional access | Instant, higher cost |
| Glacier Deep Archive | Multi-year cold archive | 12 hours |

In [ ]:
# S3 with boto3 — common data engineering patterns

import boto3

s3 = boto3.client('s3')

# List partitions to understand data coverage
response = s3.list_objects_v2(
    Bucket='telemetry-lake',
    Prefix='metrics/year=2026/month=02/',
    Delimiter='/'        # treat as folder — list "subdirectories" only
)

for prefix in response.get('CommonPrefixes', []):
    print(prefix['Prefix'])   # metrics/year=2026/month=02/day=01/, day=02/, ...

# Upload a Parquet file with correct partition path
import pandas as pd
import io

df = pd.DataFrame({'server_id': ['srv-01'], 'cpu_pct': [75.2]})

buffer = io.BytesIO()
df.to_parquet(buffer, index=False)
buffer.seek(0)

s3.put_object(
    Bucket='telemetry-lake',
    Key='metrics/year=2026/month=02/day=27/batch_001.parquet',
    Body=buffer.getvalue()
)
print("Uploaded to S3 with partition path")

## Glue — Catalog + ETL

AWS Glue has two distinct functions:

**1. Glue Data Catalog:** A metadata store — tables, schemas, partition info. Athena reads from it. Redshift Spectrum reads from it. It's the central "yellow pages" for your data lake.

**2. Glue Crawlers:** Point at an S3 path → crawler scans the data → automatically creates/updates tables in the catalog. No schema definition needed for well-structured Parquet/CSV.

**3. Glue Jobs:** Managed Spark (PySpark) jobs. Pay per DPU-second. No cluster to spin up. Timeout-based cleanup.

In [ ]:
# Glue Job (PySpark ETL) — production template
# Runs on managed Spark. No cluster management.

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
import pyspark.sql.functions as F

args = getResolvedOptions(sys.argv, ['JOB_NAME', 'source_table', 'target_path'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

# Read from Glue Data Catalog (schema auto-detected by crawler)
datasource = glueContext.create_dynamic_frame.from_catalog(
    database="telemetry_db",
    table_name=args['source_table'],
    push_down_predicate="year='2026' and month='02'"   # partition pruning
)

# Transform: filter high-CPU records, add processing timestamp
df = datasource.toDF()
df_filtered = (df
    .filter(F.col("cpu_utilization") > 80)
    .withColumn("processed_at", F.current_timestamp())
    .withColumn("alert_tier",
        F.when(F.col("cpu_utilization") > 95, "critical")
         .when(F.col("cpu_utilization") > 80, "warning")
         .otherwise("normal"))
)

# Write back as Parquet, partitioned by date
df_filtered.write.mode("overwrite").partitionBy("year", "month", "day") \
    .parquet(args['target_path'])

job.commit()
print(f"Glue job complete. Wrote high-CPU records to {args['target_path']}")

## Athena — Serverless SQL on S3

Athena runs SQL directly on S3 files. No server, no cluster, no maintenance. **Pay per TB scanned.** This makes format and partitioning choices critical for cost.

### Cost Math

| Format | Compression | 1 TB uncompressed data | Athena query cost |
|--------|-------------|----------------------|------------------|
| CSV | None | 1 TB scanned | $5.00 |
| CSV | GZIP | ~250 GB scanned | $1.25 |
| Parquet | Snappy | ~100 GB scanned | $0.50 |
| Parquet + partitioning | Snappy | ~2 GB scanned (7-day filter) | $0.01 |

**Columnar = only read the columns you SELECT.** Parquet stores each column separately. A `SELECT server_id, avg_cpu FROM ...` on a 50-column table reads only 2 columns' worth of data.

In [ ]:
-- Athena SQL — production patterns

-- 1. Partition pruning (critical for cost)
SELECT
    server_id,
    ROUND(AVG(cpu_utilization), 2) AS avg_cpu,
    MAX(cpu_utilization)           AS peak_cpu,
    COUNT(*)                       AS sample_count
FROM telemetry_db.server_metrics
WHERE year  = '2026'
  AND month = '02'              -- these are partition columns
  AND day   BETWEEN '20' AND '28'
  AND cpu_utilization > 80
GROUP BY server_id
ORDER BY avg_cpu DESC
LIMIT 100;

-- 2. Repair partitions after new data lands (if no auto-update)
MSCK REPAIR TABLE telemetry_db.server_metrics;

-- 3. Check what partitions exist
SHOW PARTITIONS telemetry_db.server_metrics;

-- 4. CTAS (Create Table As Select) — materialize a derived table
CREATE TABLE telemetry_db.high_cpu_monthly
WITH (
    format           = 'PARQUET',
    parquet_compression = 'SNAPPY',
    partitioned_by   = ARRAY['year', 'month'],
    external_location = 's3://telemetry-lake/high-cpu-monthly/'
) AS
SELECT
    server_id, year, month,
    ROUND(AVG(cpu_utilization), 2) AS avg_cpu
FROM telemetry_db.server_metrics
WHERE cpu_utilization > 80
GROUP BY server_id, year, month;

## The Full Architecture: APM → Data Lake

```
APM Agents (6,000 servers)
       │
       ▼
Amazon Kinesis Data Firehose
  - Buffers in-flight data
  - Auto-batches and writes to S3
  - Converts to Parquet (optional, with schema registry)
       │
       ▼
S3 Data Lake  (partitioned: year/month/day/env)
       │
       ├──→ AWS Glue Crawler
       │        Scans new partitions → updates Glue Data Catalog
       │
       ├──→ Glue ETL Jobs
       │        Transform + enrich → write derived tables back to S3
       │
       └──→ Amazon Athena
                Ad-hoc SQL queries
                Cost: pay per TB scanned (use Parquet + partitions)
                      │
                      ▼
              Amazon QuickSight (BI dashboards)
              Jupyter Notebooks (data science)
              API consumers (capacity planning services)
```

**Lake Formation:** Fine-grained access control layer on top of this stack. Column-level and row-level security. Relevant in regulated industries (finance, healthcare) — at Citi, this controls who can see which servers' data.

## Interview Q&A

**Q: What is a Glue Crawler?**
A: A managed job that scans S3 (or RDS, etc.) and automatically infers schema, creating or updating tables in the Glue Data Catalog. Point it at a bucket, run it on a schedule, and your catalog stays current with new partitions automatically.

**Q: What's the difference between Glue and EMR?**
A: Glue is serverless — no cluster to manage, pay per DPU-second. EMR gives you a full Hadoop/Spark cluster you control — more flexible, better for complex Spark tuning, but more ops overhead. Glue is faster to start with; EMR wins for heavy, custom workloads.

**Q: How does Athena pricing work and why does it matter?**
A: $5 per TB scanned. Columnar format (Parquet/ORC) + compression + partitioning can reduce what Athena actually reads by 90%+. A monthly report that scans 10 TB as CSV costs $50. The same report on Parquet with partitions might scan 200 GB and cost $1. Format choice is a cost engineering decision.

**Q: What is Lake Formation and when does it matter?**
A: Fine-grained access control for your data lake — column-level and row-level security on S3/Glue/Athena. At Citi, you'd use it to ensure only the network team can see network device data, while the application team can only see application server data — without separate copies of the data.

**Q: Design a pipeline to ingest server metrics from 6,000 endpoints into S3 for daily reporting.**
A: Agents push to Kinesis Firehose → Firehose batches and writes Parquet to S3 (partitioned by date + env) → Glue Crawler updates catalog → daily Glue job aggregates to summary tables → Athena for ad-hoc + QuickSight for dashboards. CloudWatch Events triggers daily Glue job. S3 lifecycle rules archive data older than 90 days to Glacier.

## The Citi Angle

**This stack solves exactly the Citi problem.** Managing 4 different APM tools (CA APM, AppDynamics, Dynatrace, BMC TrueSight) meant 4 different data formats, 4 different schemas, no unified view. The AWS data platform pattern solves this:

1. **Normalize:** Each APM tool's export goes through a Glue job that maps to a common schema
2. **Land:** All normalized data lands in S3 under the same partition structure
3. **Query:** Athena queries across all tools transparently — "show me the top 10 CPU consumers regardless of which APM monitors them"

**The quantified win:**
- Before: Monthly capacity report required 2 analyst-days of manual data collection and Excel aggregation
- After: Athena query + QuickSight dashboard → 2 hours automated, daily refresh

**Interview line:** *"At Citi, unifying 4 APM data sources into a single S3-based data lake with Athena on top was the architectural north star. The Glue + Athena stack gave us a single SQL interface across all our observability data. That's the same pattern I'd use for any APM ingestion problem."*